# 01 – HDFS dataset exploration

Placeholder: explore raw HDFS logs, label distribution, and basic stats.

In [22]:
from pathlib import Path
import re
import pandas as pd

In [23]:
from pathlib import Path

# --- Find repo root (the folder that contains api/ and agent/) ---
BASE = Path.cwd().resolve()
for _ in range(6):
    if (BASE / "api").exists() and (BASE / "agent").exists():
        break
    BASE = BASE.parent
REPO_ROOT = BASE

# --- Support BOTH folder names ---
ML_DIR = None
for cand in ["machine_learning", "machine learning"]:
    p = REPO_ROOT / cand
    if p.exists():
        ML_DIR = p
        break

if ML_DIR is None:
    raise FileNotFoundError(
        f"Could not find ML folder. Expected one of: "
        f"{REPO_ROOT/'machine_learning'} or {REPO_ROOT/'machine learning'}"
    )

RAW_DIR = ML_DIR / "data" / "raw"
PROCESSED_DIR = ML_DIR / "data" / "processed"
FEATURES_DIR = ML_DIR / "features"

# Create outputs if missing
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# Input file
LOG_FILE = RAW_DIR / "HDFS.log"
if not LOG_FILE.exists():
    raise FileNotFoundError(f"HDFS.log not found at: {LOG_FILE}")

print("REPO_ROOT:", REPO_ROOT)
print("ML_DIR:", ML_DIR)
print("LOG_FILE:", LOG_FILE)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("FEATURES_DIR:", FEATURES_DIR)

REPO_ROOT: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent
ML_DIR: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning
LOG_FILE: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/data/raw/HDFS.log
PROCESSED_DIR: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/data/processed
FEATURES_DIR: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/features


In [24]:
RAW_LOG = REPO_ROOT / "machine_learning" / "data" / "raw" / "HDFS.log"
PROCESSED_DIR = REPO_ROOT / "machine_learning" / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [25]:
if not RAW_LOG.exists():
    raise FileNotFoundError(f"HDFS log not found at: {RAW_LOG}")

In [26]:
LOG_RE = re.compile(
    r"^(?P<date>\d{6})\s+"
    r"(?P<time>\d{6})\s+"
    r"(?P<pid>\d+)\s+"
    r"(?P<level>[A-Z]+)\s+"
    r"(?P<component>[^:]+):\s+"
    r"(?P<message>.*)$"
)

In [27]:
# Parse log file with regex
rows = []
unmatched = 0
with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        m = LOG_RE.match(line)
        if m:
            rows.append(m.groupdict())
        else:
            unmatched += 1

In [28]:
df = pd.DataFrame(rows)

print("Parsed rows:", len(df))
print("Unmatched lines:", unmatched)
display(df.head())

Parsed rows: 2000
Unmatched lines: 0


,date,time,pid,level,component,message
0,081109,203615,148,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_38865049064139...
1,081109,203807,222,INFO,dfs.DataNode$PacketResponder,PacketResponder 0 for block blk_-6952295868487...
2,081109,204005,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.addStoredBlock: blockMap upd...
3,081109,204015,308,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_82291938032499...
4,081109,204106,329,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_-6670958622368...


In [29]:
STRUCTURED_PATH = PROCESSED_DIR / "hdfs_structured.parquet"
df.to_parquet(STRUCTURED_PATH, index=False)
print("Saved structured logs to:", STRUCTURED_PATH)

Saved structured logs to: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/data/processed/hdfs_structured.parquet
